## Project: 멋진 챗봇 만들기

### Step 1. 데이터 다운로드

In [1]:
import numpy
import pandas
import torch
import nltk
import gensim

print(numpy.__version__)
print(pandas.__version__)
print(torch.__version__)
print(nltk.__version__)
print(gensim.__version__)

2.2.6
2.3.0
2.7.1+cu118
3.9.3
4.4.0


In [2]:
import pandas as pd

data = pd.read_csv("ChatbotData.csv")

questions = data["Q"].tolist()
answers = data["A"].tolist()

### Step 2. 데이터 정제

In [3]:
import re

def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    
    # 영어, 한글, 숫자, 주요 문장부호(.?!,) 제외 제거
    sentence = re.sub(r"[^a-zA-Z가-힣0-9?.!,]+", " ", sentence)
    
    return sentence

In [4]:
### 테스트
print(preprocess_sentence("안녕!! 반가워요ㅎㅎㅎㅎ"))
print(preprocess_sentence("HELLO!! I'm ChatGPT :)"))

안녕!! 반가워요 
hello!! i m chatgpt 


In [5]:
# 데이터 전체 적용
questions = [preprocess_sentence(q) for q in questions]
answers = [preprocess_sentence(a) for a in answers]

### Step 3. 데이터 토큰화

In [7]:
def build_corpus(src, tgt, tokenize, max_len=15):
    
    src_corpus = []
    tgt_corpus = []
    
    src_set = set()
    tgt_set = set()
    
    for s, t in zip(src, tgt):
        
        # Step1 정제
        s = preprocess_sentence(s)
        t = preprocess_sentence(t)
        
        # Step2 토큰화
        s_tokens = tokenize(s)
        t_tokens = tokenize(t)
        
        # Step3 길이 제한
        if len(s_tokens) > max_len or len(t_tokens) > max_len:
            continue
        
        # Step4 중복 제거 (각각 검사)
        if s in src_set or t in tgt_set:
            continue
        
        src_set.add(s)
        tgt_set.add(t)
        
        src_corpus.append(s_tokens)
        tgt_corpus.append(t_tokens)
    
    return src_corpus, tgt_corpus

In [10]:
from konlpy.tag import Mecab

mecab = Mecab()

que_corpus, ans_corpus = build_corpus(
    questions,
    answers,
    mecab.morphs
)

In [11]:
# 결과 확인
print(len(que_corpus))
print(que_corpus[:3])
print(ans_corpus[:3])

7142
[['12', '시', '땡', '!'], ['1', '지망', '학교', '떨어졌', '어'], ['3', '박', '4', '일', '놀', '러', '가', '고', '싶', '다']]
[['하루', '가', '또', '가', '네요', '.'], ['위로', '해', '드립니다', '.'], ['여행', '은', '언제나', '좋', '죠', '.']]


### Step 4. Augmentation

In [20]:
# ko.bin 로드 (gensim) - ko.tsv 포맷에 맞게 “멀티라인 벡터” 복구 로더
import numpy as np
from gensim.models import KeyedVectors

def load_kyubyong_multiline_tsv(path, limit=None):
    words = []
    vecs = []
    dim = None
    
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        buf = ""
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            
            # 레코드 시작: "index \t word \t [ ... "
            if buf == "":
                # 탭 기반 우선
                parts = line.split("\t")
                if len(parts) >= 3:
                    idx, word, rest = parts[0], parts[1], "\t".join(parts[2:])
                    buf = f"{word}\t{rest}"
                else:
                    # 공백 기반 fallback
                    parts = line.split(maxsplit=2)
                    if len(parts) < 3:
                        continue
                    idx, word, rest = parts[0], parts[1], parts[2]
                    buf = f"{word}\t{rest}"
            else:
                # 벡터가 다음 줄로 이어진 경우
                buf += " " + line.strip()
            
            # 레코드 종료 조건: ']' 포함
            if "]" in buf:
                try:
                    word, vec_part = buf.split("\t", 1)
                    vec_str = vec_part.strip()
                    # '[' ~ ']' 사이만 추출
                    l = vec_str.find("[")
                    r = vec_str.rfind("]")
                    if l == -1 or r == -1 or r <= l:
                        buf = ""
                        continue
                    vec_str = vec_str[l+1:r]
                    
                    vec = np.fromstring(vec_str, sep=" ")
                    if vec.size == 0:
                        buf = ""
                        continue
                    if dim is None:
                        dim = vec.size
                    if vec.size != dim:
                        # 깨진 레코드 스킵
                        buf = ""
                        continue
                    
                    words.append(word)
                    vecs.append(vec)
                    if limit is not None and len(words) >= limit:
                        break
                finally:
                    buf = ""
    
    kv = KeyedVectors(vector_size=dim)
    kv.add_vectors(words, np.vstack(vecs))
    return kv

wv_ko = load_kyubyong_multiline_tsv("ko.tsv")
print("loaded:", len(wv_ko), "dim:", wv_ko.vector_size)
print(wv_ko.most_similar("사랑", topn=5))

loaded: 30185 dim: 200
[('슬픔', 0.7216663360595703), ('행복', 0.6759076714515686), ('절망', 0.6468985676765442), ('기쁨', 0.6458414196968079), ('이별', 0.6334798336029053)]


In [21]:
# Augmentation (Lexical Substitution) — 데이터 3배 만들기
import random
import re

def lexical_sub(tokens, wv, topn=20):
    """
    tokens(list[str])에서 치환 가능한 단어 1개를 유사어로 대체
    """
    if not tokens:
        return tokens
    
    candidates = []
    for i, tok in enumerate(tokens):
        # 벡터에 있고, 글자/숫자 토큰만
        if tok in wv and re.fullmatch(r"[가-힣a-zA-Z0-9]+", tok):
            # 너무 짧은 토큰은 치환 효과가 약해서 제외(선택)
            if len(tok) >= 2:
                candidates.append(i)

    if not candidates:
        return tokens
    
    idx = random.choice(candidates)
    src_word = tokens[idx]
    
    for cand, _ in wv.most_similar(src_word, topn=topn):
        if cand != src_word and re.fullmatch(r"[가-힣a-zA-Z0-9]+", cand):
            out = tokens.copy()
            out[idx] = cand
            return out
    
    return tokens

# (증강Q, 원A) + (원Q, 원A) + (원Q, 증강A) => 약 3배
aug_que = [lexical_sub(q, wv_ko) for q in que_corpus]
aug_ans = [lexical_sub(a, wv_ko) for a in ans_corpus]

que_corpus_aug = que_corpus + aug_que + que_corpus
ans_corpus_aug = ans_corpus + ans_corpus + aug_ans

print("원본:", len(que_corpus), "증강:", len(que_corpus_aug))
print("예시(원Q):", que_corpus[0])
print("예시(증강Q):", aug_que[0])

원본: 7142 증강: 21426
예시(원Q): ['12', '시', '땡', '!']
예시(증강Q): ['12', '시', '땡', '!']


### Step 5. 데이터 벡터화

In [24]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

START_TOKEN = "<start>"
END_TOKEN   = "<end>"
OOV_TOKEN   = "<unk>"

# 1) ans_corpus 전체에 <start>, <end> 추가
ans_corpus_se = [[START_TOKEN] + sent + [END_TOKEN] for sent in ans_corpus_aug]

# (선택) que도 같이 붙이면 보통 더 안정적임 (사전 공유 목적)
que_corpus_se = [[START_TOKEN] + sent + [END_TOKEN] for sent in que_corpus_aug]

# 2) que + ans 결합해서 단어 사전 구축 (Embedding 공유 목적)
tokenizer = Tokenizer(filters="", oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(que_corpus_se + ans_corpus_se)

vocab_size = len(tokenizer.word_index) + 1
print("vocab_size:", vocab_size)

# 3) 벡터화 (정수 인코딩)
enc_train = tokenizer.texts_to_sequences(que_corpus_se)   # encoder input
dec_train = tokenizer.texts_to_sequences(ans_corpus_se)   # decoder input/target 만들기 전 단계

# 4) 길이 통일(패딩) - max_len은 Step3에서 쓴 값 기준으로 맞추는 게 안전
MAX_LEN = 15  # 너 Step3에서 쓴 max_len 값으로 맞춰줘

enc_train = pad_sequences(enc_train, maxlen=MAX_LEN, padding="post")
dec_train = pad_sequences(dec_train, maxlen=MAX_LEN, padding="post")

print("enc_train shape:", enc_train.shape)
print("dec_train shape:", dec_train.shape)

# 5) 간단 확인
print("예시(원 ans):", ans_corpus_aug[0])
print("예시(<start>/<end> 추가):", ans_corpus_se[0])
print("예시(dec_train[0]):", dec_train[0])

2026-03-04 06:27:02.376413: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-04 06:27:04.043361: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


vocab_size: 7121
enc_train shape: (21426, 15)
dec_train shape: (21426, 15)
예시(원 ans): ['하루', '가', '또', '가', '네요', '.']
예시(<start>/<end> 추가): ['<start>', '하루', '가', '또', '가', '네요', '.', '<end>']
예시(dec_train[0]): [  2 306   9 119   9  40   4   3   0   0   0   0   0   0   0]


### Step 6. 훈련하기

In [27]:
transformer = Transformer(
    n_layers=1,
    d_model=128,
    n_heads=4,
    d_ff=512,
    input_vocab_size=vocab_size,
    target_vocab_size=vocab_size,
    dropout=0.3
)

NameError: name 'Transformer' is not defined